In [ ]:
import pandas as pd
#census = pd.read_csv('census_and_LODES_data_wide.csv')

In [ ]:
zillow_df = pd.read_csv('ZillowHousingData.csv')
zip_tract_crosswalk_df = pd.read_csv('zip_tract_crosswalk.csv')

# Zillow Data Manipulation

1. Merge with zip_tract_crosswalk
2. Create 5-year home value averages


In [ ]:
# Convert 'RegionName' to numeric to match 'zip_code' for merging
zillow_df['RegionName'] = pd.to_numeric(zillow_df['RegionName'], errors='coerce').astype('Int64')

# Merge the dataframes
zillow_merged_df = pd.merge(
    zillow_df,
    zip_tract_crosswalk_df,
    left_on='RegionName',
    right_on='zip_code',
    how='right'
)

print("Shape of merged Zillow and Crosswalk DataFrame:", zillow_merged_df.shape)

Shape of merged Zillow and Crosswalk DataFrame: (7049, 326)


In [ ]:
# Identify specific January columns for the requested years
january_years = [2006, 2010, 2011, 2015, 2016, 2020, 2024]

# Add these specific January housing values to zillow_merged_df
for year in january_years:
    col_name = f'{year}-01-31'
    new_col_name = f'housing_value_jan_{year}'
    if col_name in zillow_merged_df.columns:
        zillow_merged_df[new_col_name] = zillow_merged_df[col_name]
    else:
        print(f"Warning: Column '{col_name}' not found in zillow_merged_df. Assigning NaN.")
        zillow_merged_df[new_col_name] = float('nan') # Assign NaN if column not found

print("Shape of zillow_merged_df after adding specific January housing value columns:", zillow_merged_df.shape)
print("First 5 rows with new January housing value columns:")
display(zillow_merged_df[['tract_id'] + [f'housing_value_jan_{year}' for year in january_years]].head())

Shape of zillow_merged_df after adding specific January housing value columns: (7049, 333)
First 5 rows with new January housing value columns:


,tract_id,housing_value_jan_2006,housing_value_jan_2010,housing_value_jan_2011,housing_value_jan_2015,housing_value_jan_2016,housing_value_jan_2020,housing_value_jan_2024
0,6059062629,972940.058344,799413.885200,791275.204269,1.044569e+06,1.000089e+06,1.197060e+06,1.970082e+06
1,6059062630,972940.058344,799413.885200,791275.204269,1.044569e+06,1.000089e+06,1.197060e+06,1.970082e+06
2,6059062631,972940.058344,799413.885200,791275.204269,1.044569e+06,1.000089e+06,1.197060e+06,1.970082e+06
3,6059062632,592101.924706,414651.051345,395787.818326,5.061658e+05,5.066049e+05,6.092836e+05,9.245512e+05
4,6059062633,592101.924706,414651.051345,395787.818326,5.061658e+05,5.066049e+05,6.092836e+05,9.245512e+05


In [ ]:
# Create a new DataFrame with only the specified columns (tract_id, zip_code, and new January values)
zillow_january_housing_averages_df = zillow_merged_df[[
    'tract_id',
    'zip_code',
    'housing_value_jan_2006',
    'housing_value_jan_2010',
    'housing_value_jan_2011',
    'housing_value_jan_2015',
    'housing_value_jan_2016',
    'housing_value_jan_2020',
    'housing_value_jan_2024'
]].copy()

print("Shape of zillow_january_housing_averages_df:", zillow_january_housing_averages_df.shape)
print("First 5 rows of the new DataFrame with specific January home values:")
display(zillow_january_housing_averages_df.head())

Shape of zillow_january_housing_averages_df: (7049, 9)
First 5 rows of the new DataFrame with specific January home values:


,tract_id,zip_code,housing_value_jan_2006,housing_value_jan_2010,housing_value_jan_2011,housing_value_jan_2015,housing_value_jan_2016,housing_value_jan_2020,housing_value_jan_2024
0,6059062629,92603,972940.058344,799413.885200,791275.204269,1.044569e+06,1.000089e+06,1.197060e+06,1.970082e+06
1,6059062630,92603,972940.058344,799413.885200,791275.204269,1.044569e+06,1.000089e+06,1.197060e+06,1.970082e+06
2,6059062631,92603,972940.058344,799413.885200,791275.204269,1.044569e+06,1.000089e+06,1.197060e+06,1.970082e+06
3,6059062632,92656,592101.924706,414651.051345,395787.818326,5.061658e+05,5.066049e+05,6.092836e+05,9.245512e+05
4,6059062633,92656,592101.924706,414651.051345,395787.818326,5.061658e+05,5.066049e+05,6.092836e+05,9.245512e+05


In [ ]:
# ZIP codes for each city
SF_ZIPS = {
    "94102","94103","94104","94105","94107","94108","94109","94110",
    "94111","94112","94114","94115","94116","94117","94118","94119",
    "94121","94122","94123","94124","94127","94129","94130","94131",
    "94132","94133","94134","94158"
}

OAKLAND_ZIPS = {
    "94601","94602","94603","94604","94605","94606","94607","94608",
    "94609","94610","94611","94612","94613","94618","94619","94621"
}

SAN_JOSE_ZIPS = {
    "95101","95110","95111","95112","95113","95116","95117","95118",
    "95119","95120","95121","95122","95123","95124","95125","95126",
    "95127","95128","95129","95130","95131","95132","95133","95134",
    "95135","95136","95138","95139","95148"
}

# Normalize ZIP column to 5-digit string
zillow_merged_df["zip_code"] = zillow_merged_df["zip_code"].astype(str).str.zfill(5)

# Create indicator variables
zillow_merged_df["city_sf"]       = zillow_merged_df["zip_code"].isin(SF_ZIPS).astype(int)
zillow_merged_df["city_oakland"]  = zillow_merged_df["zip_code"].isin(OAKLAND_ZIPS).astype(int)
zillow_merged_df["city_san_jose"] = zillow_merged_df["zip_code"].isin(SAN_JOSE_ZIPS).astype(int)

# Create a single city label column for readability
def label_city(row):
    if row["city_sf"]:       return "San Francisco"
    if row["city_oakland"]:  return "Oakland"
    if row["city_san_jose"]: return "San Jose"
    return "Other"

zillow_merged_df["city"] = zillow_merged_df.apply(label_city, axis=1)

# Verify counts
print(zillow_merged_df["city"].value_counts())
print(f"\nRows flagged as a city: {(zillow_merged_df['city'] != 'Other').sum():,}")
print(f"Rows labeled Other:     {(zillow_merged_df['city'] == 'Other').sum():,}")

city
Other            6578
San Jose          178
San Francisco     167
Oakland           126
Name: count, dtype: int64

Rows flagged as a city: 471
Rows labeled Other:     6,578


In [ ]:
from google.colab import files
zillow_merged_df.to_csv('zillow_merged_df.csv', index=False)
files.download('zillow_merged_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Census Data Manipulation

1. Clean to remove rows without data
2. Separate 2022 data for later merge with testing data
3. Separate Census Data to create training and validating sets

In [ ]:
print("Shape of census before cleaning:", census.shape)
census_cleaned = census.dropna()
print("Shape of census after cleaning:", census_cleaned.shape)

Shape of census before cleaning: (1590, 79)
Shape of census after cleaning: (1109, 79)


In [ ]:
columns_2022 = [col for col in census_cleaned.columns if '_2022' in col]
columns_for_2022_csv = ['tract_id'] + columns_2022

# Create dataframe with only 2022 features and tract_id
df_LODES_2022_features = census_cleaned[columns_for_2022_csv]
df_LODES_2022_features.to_csv('LODES_2022_features_test_set.csv', index=False)

# Create dataframe without 2022 features but including tract_id and other non-year columns
non_2022_cols = [col for col in census_cleaned.columns if '_2022' not in col]
census_cleaned_no_2022 = census_cleaned[non_2022_cols]

print("Created 'df_LODES_2022_features.csv' with the following columns and first 5 rows:")
display(df_LODES_2022_features.head())

print("\nCreated 'census_cleaned_no_2022' (without 2022 features). Shape and first 5 rows:")
print("Shape of census_cleaned_no_2022:", census_cleaned_no_2022.shape)
display(census_cleaned_no_2022.head())

Created 'df_LODES_2022_features.csv' with the following columns and first 5 rows:


,tract_id,jobs_arts_2022,jobs_food_2022,jobs_total_2022,arts_density_2022,food_density_2022,gentrify_density_2022,arts_job_share_2022,food_job_share_2022,area_sqmi_2022
0,6001400100,0.0,17.0,285.0,0.000000,6.396060,6.396060,0.000000,0.059649,2.657886
1,6001400200,0.0,232.0,1156.0,0.000000,1009.004334,1009.004334,0.000000,0.200692,0.229930
2,6001400300,15.0,677.0,2022.0,35.160786,1586.923454,1622.084240,0.007418,0.334817,0.426612
3,6001400400,14.0,339.0,1036.0,51.528280,1247.720484,1299.248764,0.013514,0.327220,0.271695
4,6001400500,30.0,56.0,348.0,132.117304,246.618968,378.736273,0.086207,0.160920,0.227071



Created 'census_cleaned_no_2022' (without 2022 features). Shape and first 5 rows:
Shape of census_cleaned_no_2022: (1109, 70)


,tract_id,population_2010,median_income_2010,median_rent_2010,median_home_value_2010,share_college_2010,share_25_34_2010,homeownership_rate_2010,vacancy_rate_2010,share_pre1940_2010,...,area_sqmi_2015,jobs_arts_2020,jobs_food_2020,jobs_total_2020,arts_density_2020,food_density_2020,gentrify_density_2020,arts_job_share_2020,food_job_share_2020,area_sqmi_2020
0,6001400100,2701.0,173472.0,2001.0,1000001.0,0.841828,0.021103,0.883371,0.078363,0.090846,...,2.657886,0.0,210.0,515.0,0.000000,79.010159,79.010159,0.000000,0.407767,2.657886
1,6001400200,2050.0,95833.0,1342.0,902400.0,0.810191,0.085854,0.683921,0.036093,0.801486,...,0.229930,0.0,153.0,1032.0,0.000000,665.420962,665.420962,0.000000,0.148256,0.229930
2,6001400300,4719.0,52314.0,998.0,740000.0,0.674725,0.093028,0.424652,0.069552,0.616352,...,0.426612,58.0,573.0,2070.0,135.955037,1343.142008,1479.097045,0.028019,0.276812,0.426612
3,6001400400,3930.0,79071.0,1387.0,768000.0,0.800574,0.160051,0.482850,0.029698,0.695341,...,0.271695,6.0,88.0,755.0,22.083548,323.892043,345.975591,0.007947,0.116556,0.271695
4,6001400500,3506.0,52076.0,1005.0,592100.0,0.552023,0.233600,0.419533,0.047396,0.711527,...,0.227071,9.0,31.0,302.0,39.635191,136.521215,176.156406,0.029801,0.102649,0.227071


In [ ]:
import pandas as pd
import numpy as np

# Assuming census_cleaned_no_2022 is available from the previous cell

columns_2010 = [col for col in census_cleaned_no_2022.columns if '_2010' in col]
columns_2015 = [col for col in census_cleaned_no_2022.columns if '_2015' in col]
columns_2020 = [col for col in census_cleaned_no_2022.columns if '_2020' in col]

non_year_columns_final = ['tract_id']

# --- Process 2010 data for training ---
df_2010 = census_cleaned_no_2022[non_year_columns_final + columns_2010].copy()
df_2010.rename(columns=lambda x: x.replace('_2010', '') if '_2010' in x else x, inplace=True)
df_2010['year'] = 2010

# --- Process 2015 data for training ---
df_2015 = census_cleaned_no_2022[non_year_columns_final + columns_2015].copy()
df_2015.rename(columns=lambda x: x.replace('_2015', '') if '_2015' in x else x, inplace=True)
df_2015['year'] = 2015

# Concatenate 2010 and 2015 data to form the new census_and_LODES_training
census_and_LODES_training = pd.concat([df_2010, df_2015], ignore_index=True)

# Create the 2020 dataset (making it consistent with the long format)
census_and_LODES_validate = census_cleaned_no_2022[non_year_columns_final + columns_2020].copy()
census_and_LODES_validate.rename(columns=lambda x: x.replace('_2020', '') if '_2020' in x else x, inplace=True)
census_and_LODES_validate['year'] = 2020

# Save to CSV files
census_and_LODES_training.to_csv('census_and_LODES_training.csv', index=False)
census_and_LODES_validate.to_csv('census_and_LODES_validate.csv', index=False)

print("Re-created 'census_training.csv' and 'census_validation.csv' with separated year data.")

print("DataFrame for training features (first 5 rows):")
display(census_and_LODES_training.head())

print("\nDataFrame for validation features (first 5 rows):")
display(census_and_LODES_validate.head())

Re-created 'census_training.csv' and 'census_validation.csv' with separated year data.
DataFrame for training features (first 5 rows):


,tract_id,population,median_income,median_rent,median_home_value,share_college,share_25_34,homeownership_rate,vacancy_rate,share_pre1940,...,jobs_arts,jobs_food,jobs_total,arts_density,food_density,gentrify_density,arts_job_share,food_job_share,area_sqmi,year
0,6001400100,2701.0,173472.0,2001.0,1000001.0,0.841828,0.021103,0.883371,0.078363,0.090846,...,23.0,290.0,13846.0,8.653494,109.109267,117.762761,0.001661,0.020945,2.657886,2010
1,6001400200,2050.0,95833.0,1342.0,902400.0,0.810191,0.085854,0.683921,0.036093,0.801486,...,2.0,153.0,1290.0,8.698313,665.420962,674.119275,0.001550,0.118605,0.229930,2010
2,6001400300,4719.0,52314.0,998.0,740000.0,0.674725,0.093028,0.424652,0.069552,0.616352,...,3.0,789.0,1966.0,7.032157,1849.457319,1856.489477,0.001526,0.401322,0.426612,2010
3,6001400400,3930.0,79071.0,1387.0,768000.0,0.800574,0.160051,0.482850,0.029698,0.695341,...,18.0,92.0,705.0,66.250645,338.614409,404.865054,0.025532,0.130496,0.271695,2010
4,6001400500,3506.0,52076.0,1005.0,592100.0,0.552023,0.233600,0.419533,0.047396,0.711527,...,0.0,69.0,411.0,0.000000,303.869800,303.869800,0.000000,0.167883,0.227071,2010



DataFrame for validation features (first 5 rows):


,tract_id,population,median_income,median_rent,median_home_value,share_college,share_25_34,homeownership_rate,vacancy_rate,share_pre1940,...,jobs_arts,jobs_food,jobs_total,arts_density,food_density,gentrify_density,arts_job_share,food_job_share,area_sqmi,year
0,6001400100,3035.0,220921.0,3501.0,1244900.0,0.875622,0.029325,0.891680,0.097094,0.038979,...,0.0,210.0,515.0,0.000000,79.010159,79.010159,0.000000,0.407767,2.657886,2020
1,6001400200,1983.0,200192.0,2442.0,1577800.0,0.856075,0.142209,0.573494,0.030374,0.074766,...,0.0,153.0,1032.0,0.000000,665.420962,665.420962,0.000000,0.148256,0.229930,2020
2,6001400300,5058.0,118695.0,1854.0,1281800.0,0.755716,0.116449,0.368747,0.095363,0.038519,...,58.0,573.0,2070.0,135.955037,1343.142008,1479.097045,0.028019,0.276812,0.426612,2020
3,6001400400,4179.0,137067.0,2097.0,1167600.0,0.762749,0.141421,0.444253,0.076433,0.061571,...,6.0,88.0,755.0,22.083548,323.892043,345.975591,0.007947,0.116556,0.271695,2020
4,6001400500,4021.0,110052.0,1905.0,982900.0,0.673625,0.170604,0.493001,0.070175,0.057159,...,9.0,31.0,302.0,39.635191,136.521215,176.156406,0.029801,0.102649,0.227071,2020


# Create Merged Training and Validation Sets (CENSUS, LODES, & Zillow combined)

In [ ]:
# Merge census_and_LODES_training with zillow_merged_df, and include the city indicator variables.
merged_training_data = pd.merge(
    census_and_LODES_training,
    zillow_merged_df[['tract_id', 'housing_value_jan_2006', 'housing_value_jan_2010', 'housing_value_jan_2011', 'housing_value_jan_2015', 'city_sf', 'city_oakland', 'city_san_jose', 'city']],
    on='tract_id',
    how='inner'
)

# Assign jan_housing_value_begin and jan_housing_value_end based on the year
merged_training_data['jan_housing_value_begin'] = merged_training_data.apply(
    lambda row: row['housing_value_jan_2006'] if row['year'] == 2010 else
                (row['housing_value_jan_2011'] if row['year'] == 2015 else None),
    axis=1
)
merged_training_data['jan_housing_value_end'] = merged_training_data.apply(
    lambda row: row['housing_value_jan_2010'] if row['year'] == 2010 else
                (row['housing_value_jan_2015'] if row['year'] == 2015 else None),
    axis=1
)

# Drop the original year-specific housing value columns
merged_training_data.drop(columns=[
    'housing_value_jan_2006', 'housing_value_jan_2010',
    'housing_value_jan_2011', 'housing_value_jan_2015'
], inplace=True)

print("Shape of the merged_training_data DataFrame:", merged_training_data.shape)
print("First 5 rows of the new merged training dataset:")
display(merged_training_data.head())

Shape of the merged_training_data DataFrame: (1744, 31)
First 5 rows of the new merged training dataset:


,tract_id,population,median_income,median_rent,median_home_value,share_college,share_25_34,homeownership_rate,vacancy_rate,share_pre1940,...,arts_job_share,food_job_share,area_sqmi,year,city_sf,city_oakland,city_san_jose,city,jan_housing_value_begin,jan_housing_value_end
0,6001400100,2701.0,173472.0,2001.0,1000001.0,0.841828,0.021103,0.883371,0.078363,0.090846,...,0.001661,0.020945,2.657886,2010,0,1,0,Oakland,941476.279318,797028.231912
1,6001400200,2050.0,95833.0,1342.0,902400.0,0.810191,0.085854,0.683921,0.036093,0.801486,...,0.001550,0.118605,0.229930,2010,0,1,0,Oakland,592873.142405,486074.368121
2,6001400300,4719.0,52314.0,998.0,740000.0,0.674725,0.093028,0.424652,0.069552,0.616352,...,0.001526,0.401322,0.426612,2010,0,1,0,Oakland,592873.142405,486074.368121
3,6001400400,3930.0,79071.0,1387.0,768000.0,0.800574,0.160051,0.482850,0.029698,0.695341,...,0.025532,0.130496,0.271695,2010,0,1,0,Oakland,592873.142405,486074.368121
4,6001400500,3506.0,52076.0,1005.0,592100.0,0.552023,0.233600,0.419533,0.047396,0.711527,...,0.000000,0.167883,0.227071,2010,0,1,0,Oakland,592873.142405,486074.368121


In [ ]:
# Merge census_and_LODES_validate (which has 2020 data) with zillow_merged_df
# to include the January 2016 and 2020 housing values, and the city indicator variables.
merged_validation_data = pd.merge(
    census_and_LODES_validate,
    zillow_merged_df[['tract_id', 'housing_value_jan_2016', 'housing_value_jan_2020', 'city_sf', 'city_oakland', 'city_san_jose', 'city']],
    on='tract_id',
    how='inner'
)

# Assign jan_housing_value_begin and jan_housing_value_end for 2020
merged_validation_data['jan_housing_value_begin'] = merged_validation_data.apply(
    lambda row: row['housing_value_jan_2016'] if row['year'] == 2020 else None,
    axis=1
)
merged_validation_data['jan_housing_value_end'] = merged_validation_data.apply(
    lambda row: row['housing_value_jan_2020'] if row['year'] == 2020 else None,
    axis=1
)

# Drop the original year-specific housing value columns
merged_validation_data.drop(columns=[
    'housing_value_jan_2016', 'housing_value_jan_2020'
], inplace=True)

print("Shape of the merged_validation_data DataFrame:", merged_validation_data.shape)
print("First 5 rows of the new merged validation dataset:")
display(merged_validation_data.head())

Shape of the merged_validation_data DataFrame: (872, 31)
First 5 rows of the new merged validation dataset:


,tract_id,population,median_income,median_rent,median_home_value,share_college,share_25_34,homeownership_rate,vacancy_rate,share_pre1940,...,arts_job_share,food_job_share,area_sqmi,year,city_sf,city_oakland,city_san_jose,city,jan_housing_value_begin,jan_housing_value_end
0,6001400100,3035.0,220921.0,3501.0,1244900.0,0.875622,0.029325,0.891680,0.097094,0.038979,...,0.000000,0.407767,2.657886,2020,0,1,0,Oakland,1.135553e+06,1.312355e+06
1,6001400200,1983.0,200192.0,2442.0,1577800.0,0.856075,0.142209,0.573494,0.030374,0.074766,...,0.000000,0.148256,0.229930,2020,0,1,0,Oakland,8.598882e+05,1.086389e+06
2,6001400300,5058.0,118695.0,1854.0,1281800.0,0.755716,0.116449,0.368747,0.095363,0.038519,...,0.028019,0.276812,0.426612,2020,0,1,0,Oakland,8.598882e+05,1.086389e+06
3,6001400400,4179.0,137067.0,2097.0,1167600.0,0.762749,0.141421,0.444253,0.076433,0.061571,...,0.007947,0.116556,0.271695,2020,0,1,0,Oakland,8.598882e+05,1.086389e+06
4,6001400500,4021.0,110052.0,1905.0,982900.0,0.673625,0.170604,0.493001,0.070175,0.057159,...,0.029801,0.102649,0.227071,2020,0,1,0,Oakland,8.598882e+05,1.086389e+06


# Create a tract_id and Description dataframe

In [ ]:
# Identify all columns in zillow_merged_df
all_zillow_cols = zillow_merged_df.columns.tolist()

# Identify columns that are housing values (either original date format or the new January columns)
housing_value_cols = [col for col in all_zillow_cols if ('-' in col and col.count('-') == 2 and col.split('-')[0].isdigit()) or 'housing_value_jan_' in col]

# Identify descriptor columns by excluding housing value columns and ensuring 'tract_id' is included
# and moving 'zip_code' next to 'tract_id'
descriptor_cols = ['tract_id', 'zip_code'] + \
    [col for col in all_zillow_cols if col not in housing_value_cols and col not in ['tract_id', 'zip_code', 'RegionID', 'RegionName']]

# Create the new DataFrame with only tract_id and descriptor columns
zillow_descriptor_df = zillow_merged_df[descriptor_cols].copy()

print("Shape of the Zillow Descriptor DataFrame:", zillow_descriptor_df.shape)
print("First 5 rows of the Zillow Descriptor DataFrame:")
display(zillow_descriptor_df.head())

Shape of the Zillow Descriptor DataFrame: (7049, 13)
First 5 rows of the Zillow Descriptor DataFrame:


,tract_id,zip_code,SizeRank,RegionType,StateName,State,City,Metro,CountyName,city_sf,city_oakland,city_san_jose,city
0,6059062629,92603,6403.0,zip,CA,CA,Irvine,"Los Angeles-Long Beach-Anaheim, CA",Orange County,0,0,0,Other
1,6059062630,92603,6403.0,zip,CA,CA,Irvine,"Los Angeles-Long Beach-Anaheim, CA",Orange County,0,0,0,Other
2,6059062631,92603,6403.0,zip,CA,CA,Irvine,"Los Angeles-Long Beach-Anaheim, CA",Orange County,0,0,0,Other
3,6059062632,92656,813.0,zip,CA,CA,Aliso Viejo,"Los Angeles-Long Beach-Anaheim, CA",Orange County,0,0,0,Other
4,6059062633,92656,813.0,zip,CA,CA,Aliso Viejo,"Los Angeles-Long Beach-Anaheim, CA",Orange County,0,0,0,Other


# Add the Gentrification Measures

1. Create City Median income column

2. Create indicator variable for each city


Create columns to identify changes in:

1. Average income

2. Appreciation in home values
  (from start to end of 5-year estimate period?)

3. Rent

4. Education percentage

### Create City Median Income Columns

In [ ]:
# Calculate city median income for each year in the training data
city_median_income_by_year = merged_training_data.groupby(['city', 'year'])['median_income'].median().reset_index()
city_median_income_by_year.rename(columns={'median_income': 'city_median_income'}, inplace=True)

# Merge city median income back into merged_training_data
merged_training_data = pd.merge(
    merged_training_data,
    city_median_income_by_year,
    on=['city', 'year'],
    how='left'
)

print("Shape of merged_training_data after adding city median income:", merged_training_data.shape)
print("First 5 rows with new city median income column (training data):")
display(merged_training_data[['tract_id', 'city', 'year', 'median_income', 'city_median_income']].head())

Shape of merged_training_data after adding city median income: (1744, 32)
First 5 rows with new city median income column (training data):


,tract_id,city,year,median_income,city_median_income
0,6001400100,Oakland,2010,173472.0,50311.0
1,6001400200,Oakland,2010,95833.0,50311.0
2,6001400300,Oakland,2010,52314.0,50311.0
3,6001400400,Oakland,2010,79071.0,50311.0
4,6001400500,Oakland,2010,52076.0,50311.0


In [ ]:
# Calculate city median income for each year in the validation data
city_median_income_by_year_validate = merged_validation_data.groupby(['city', 'year'])['median_income'].median().reset_index()
city_median_income_by_year_validate.rename(columns={'median_income': 'city_median_income'}, inplace=True)

# Merge city median income back into merged_validation_data
merged_validation_data = pd.merge(
    merged_validation_data,
    city_median_income_by_year_validate,
    on=['city', 'year'],
    how='left'
)

print("Shape of merged_validation_data after adding city median income:", merged_validation_data.shape)
print("First 5 rows with new city median income column (validation data):")
display(merged_validation_data[['tract_id', 'city', 'year', 'median_income', 'city_median_income']].head())

Shape of merged_validation_data after adding city median income: (872, 32)
First 5 rows with new city median income column (validation data):


,tract_id,city,year,median_income,city_median_income
0,6001400100,Oakland,2020,220921.0,86405.0
1,6001400200,Oakland,2020,200192.0,86405.0
2,6001400300,Oakland,2020,118695.0,86405.0
3,6001400400,Oakland,2020,137067.0,86405.0
4,6001400500,Oakland,2020,110052.0,86405.0


# Add Gentrification Indication

Which tracts do we know got gentrified, and which have not?

Criteria: Change from

# Gentrification Indicator for training data (Use 2010 to compute on 2015)

In [ ]:
import numpy as np

# Split merged_training_data into 2010 and 2015 subsets for change calculation
df_2010 = merged_training_data[merged_training_data['year'] == 2010].copy()
df_2015 = merged_training_data[merged_training_data['year'] == 2015].copy()

# Merge 2015 data back to 2010 data to align for calculations, ensuring tract_id alignment
# This creates a temporary DataFrame to hold both 2010 and 2015 values for comparison
calc_df = pd.merge(
    df_2010,
    df_2015[['tract_id', 'median_income', 'median_rent', 'median_home_value', 'share_college', 'city_median_income']],
    on='tract_id',
    how='inner',
    suffixes=('_2010', '_2015')
)

# ── STEP 1: COMPUTE CHANGES IN EACH METRIC (2010 → 2015) ─────────────────────
# Use 'jan_housing_value_begin' and 'jan_housing_value_end' for home value change from the original merged_training_data
calc_df['income_change']    = (calc_df['median_income_2015']    - calc_df['median_income_2010'])    / calc_df['median_income_2010']
calc_df['rent_change']      = (calc_df['median_rent_2015']      - calc_df['median_rent_2010'])      / calc_df['median_rent_2010']
calc_df['homevalue_change'] = (calc_df['median_home_value_2015'] - calc_df['median_home_value_2010']) / calc_df['median_home_value_2010']
calc_df['college_change']   = (calc_df['share_college_2015']    - calc_df['share_college_2010'])    / calc_df['share_college_2010'].replace(0, np.nan)

# ── STEP 2: FLAG TOP QUARTILE CHANGE WITHIN EACH CITY ────────────────────────
# For each metric, a tract qualifies if its change is above the 75th percentile
# among tracts in the same city

change_cols = ["income_change", "rent_change", "homevalue_change", "college_change"]

for col in change_cols:
    flag_col = f"{col}_top_q"
    # Compute 75th percentile within city group
    city_75th = calc_df.groupby("city")['income_change'].transform(lambda x: x.quantile(0.75))
    calc_df[flag_col] = (calc_df[col] > city_75th).astype(int)

# ── STEP 3: COUNT HOW MANY CRITERIA ARE MET ───────────────────────────────────

flag_cols = [f"{col}_top_q" for col in change_cols]
calc_df["criteria_met"] = calc_df[flag_cols].sum(axis=1)

# ── STEP 4: APPLY GENTRIFICATION DEFINITION ───────────────────────────────────
# Condition 1: below median city income in 2010 (eligible tract)
# Condition 2: top quartile in at least 3 of 4 metrics

calc_df["below_city_median_income"] = (calc_df["median_income_2010"] < calc_df["city_median_income_2010"]).astype(int)
calc_df["gentrified"] = (
    (calc_df["below_city_median_income"] == 1) &
    (calc_df["criteria_met"] >= 3)
).astype(int)

# Add the 'gentrified', 'below_city_median_income' and 'criteria_met' columns back to the 2015 part of merged_training_data
merged_training_data = pd.merge(
    merged_training_data,
    calc_df[['tract_id', 'gentrified', 'below_city_median_income', 'criteria_met']],
    on='tract_id',
    how='left'
)

# ── STEP 5: SUMMARY ───────────────────────────────────────────────────────────

# Filter for 2015 data to get accurate summary, as gentrification is computed for 2015 based on 2010
summary_df = merged_training_data[merged_training_data['year'] == 2015]

print("=== GENTRIFICATION LABELS ===\n")
print(f"Total tracts:              {len(summary_df):,}")
print(f"Eligible (below median):   {summary_df['below_city_median_income'].sum():,}")
print(f"Gentrified:                {summary_df['gentrified'].sum():,}")
print(f"Not gentrified (eligible): {(summary_df['below_city_median_income'] - summary_df['gentrified']).sum():,}")
print()
print("By city:")
print(summary_df.groupby("city")[["below_city_median_income", "gentrified"]].sum().to_string())
print()
print("Criteria breakdown (among eligible tracts):")
eligible = summary_df[summary_df["below_city_median_income"] == 1]
print(eligible["criteria_met"].value_counts().sort_index().rename("tract count"))

=== GENTRIFICATION LABELS ===

Total tracts:              872
Eligible (below median):   435
Gentrified:                32
Not gentrified (eligible): 403

By city:
               below_city_median_income  gentrified
city                                               
Oakland                              54           1
Other                               262          24
San Francisco                        49           2
San Jose                             70           5

Criteria breakdown (among eligible tracts):
criteria_met
0    148
1    171
2     84
3     29
4      3
Name: tract count, dtype: int64


# Gentrification Indicator for training data (Use 2015 to compute on 2020)

In [ ]:
import numpy as np

# Get 2020 data for the 'end' state from merged_validation_data
df_2020 = merged_validation_data[merged_validation_data['year'] == 2020].copy()

# Get 2015 data for the 'begin' state from merged_training_data (which includes city_median_income)
df_2015_base = merged_training_data[merged_training_data['year'] == 2015].copy()

# Merge 2015 data into the 2020 data for change calculation
calc_df_validate = pd.merge(
    df_2020,
    df_2015_base[['tract_id', 'median_income', 'median_rent', 'median_home_value', 'share_college', 'city_median_income']],
    on='tract_id',
    how='inner',
    suffixes=('_2020', '_2015')
)

# ── STEP 1: COMPUTE CHANGES IN EACH METRIC (2015 → 2020) FOR VALIDATION DATA ─────────────────────
# Use 'jan_housing_value_begin' and 'jan_housing_value_end' which are already in df_2020 (representing 2016 and 2020)
calc_df_validate["income_change"]    = (calc_df_validate["median_income_2020"]    - calc_df_validate["median_income_2015"])    / calc_df_validate["median_income_2015"]
calc_df_validate["rent_change"]      = (calc_df_validate["median_rent_2020"]      - calc_df_validate["median_rent_2015"])      / calc_df_validate["median_rent_2015"]
calc_df_validate["homevalue_change"] = (calc_df_validate["median_home_value_2020"] - calc_df_validate["median_home_value_2015"]) / calc_df_validate["median_home_value_2015"]
calc_df_validate["college_change"]   = (calc_df_validate["share_college_2020"]    - calc_df_validate["share_college_2015"])    / calc_df_validate["share_college_2015"].replace(0, np.nan)

# ── STEP 2: FLAG TOP QUARTILE CHANGE WITHIN EACH CITY FOR VALIDATION DATA ────────────────────────
change_cols = ["income_change", "rent_change", "homevalue_change", "college_change"]

for col in change_cols:
    flag_col = f"{col}_top_q"
    # Compute 75th percentile within city group
    city_75th = calc_df_validate.groupby("city")['income_change'].transform(lambda x: x.quantile(0.75))
    calc_df_validate[flag_col] = (calc_df_validate[col] > city_75th).astype(int)

# ── STEP 3: COUNT HOW MANY CRITERIA ARE MET FOR VALIDATION DATA ───────────────────────────────────

flag_cols = [f"{col}_top_q" for col in change_cols]
calc_df_validate["criteria_met"] = calc_df_validate[flag_cols].sum(axis=1)

# ── STEP 4: APPLY GENTRIFICATION DEFINITION FOR VALIDATION DATA ───────────────────────────────────
# Condition 1: below median city income in 2015 (eligible tract)
# Condition 2: top quartile in at least 2 of 4 metrics

calc_df_validate["below_city_median_income"] = (
    calc_df_validate["median_income_2015"] < calc_df_validate["city_median_income_2015"]
).astype(int)

calc_df_validate["gentrified"] = (
    (calc_df_validate["below_city_median_income"] == 1) &
    (calc_df_validate["criteria_met"] >= 3)
).astype(int)

# Add the final 'gentrified', 'below_city_median_income' and 'criteria_met' columns to the actual merged_validation_data
merged_validation_data = pd.merge(
    merged_validation_data,
    calc_df_validate[['tract_id', 'gentrified', 'below_city_median_income', 'criteria_met']],
    on='tract_id',
    how='left'
)

# ── STEP 5: SUMMARY FOR VALIDATION DATA ───────────────────────────────────────────────────────────

# Filter for 2020 data to get accurate summary, as gentrification is computed for 2020 based on 2015
summary_df_validate = merged_validation_data[merged_validation_data['year'] == 2020]

print("\n=== VALIDATION GENTRIFICATION LABELS (2015→2020) ===\n")
print(f"Total tracts:              {len(summary_df_validate):,}")
print(f"Eligible (below median):   {summary_df_validate['below_city_median_income'].sum():,}")
print(f"Gentrified:                {summary_df_validate['gentrified'].sum():,}")
print(f"Not gentrified (eligible): {(summary_df_validate['below_city_median_income'] - summary_df_validate['gentrified']).sum():,}")
print()
print("By city (Validation Data):")
print(summary_df_validate.groupby("city")[["below_city_median_income", "gentrified"]].sum().to_string())
print()
print("Criteria breakdown (among eligible tracts - Validation Data):")
eligible_validation = summary_df_validate[summary_df_validate["below_city_median_income"] == 1]
print(eligible_validation["criteria_met"].value_counts().sort_index().rename("tract count"))


=== VALIDATION GENTRIFICATION LABELS (2015→2020) ===

Total tracts:              872
Eligible (below median):   435
Gentrified:                59
Not gentrified (eligible): 376

By city (Validation Data):
               below_city_median_income  gentrified
city                                               
Oakland                              54           7
Other                               262          42
San Francisco                        49           1
San Jose                             70           9

Criteria breakdown (among eligible tracts - Validation Data):
criteria_met
0     84
1    174
2    118
3     51
4      8
Name: tract count, dtype: int64


In [ ]:
merged_training_data.to_csv('merged_training_data_long.csv', index=False)
merged_validation_data.to_csv('merged_validation_data_long.csv', index=False)
from google.colab import files
files.download('merged_training_data_long.csv')
files.download('merged_validation_data_long.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>